In [ ]:
#1.读取解析yaml
import yaml
import numpy as np

def parse_attack_chain(yaml_path):
    with open(yaml_path) as f:
        data = yaml.safe_load(f)
    
    sequence = []
    for step in data['emulation_plan_details']:
        tactic = step['tactic']
        technique = step['technique']['attack_id']
        platforms = list(step['platforms'].keys())
        cmd_length = len(step['executors'][0]['command']) if 'command' in step else 0
        
        sequence.append({
            'tactic': tactic,
            'technique': technique,
            'platforms': platforms,
            'cmd_complexity': cmd_length // 100  # 命令复杂度分级
        })
    return sequence

In [ ]:
from sklearn.preprocessing import LabelEncoder
import torch
import numpy as np

class FeatureEncoder:
    def __init__(self):
        self.tactic_encoder = LabelEncoder()
        self.technique_encoder = LabelEncoder()
        self.platform_encoder = LabelEncoder()
        
    def fit(self, sequences):
        # 收集所有唯一值
        tactics, techniques, platforms, cmd_complexities = [], [], [], []
        for seq in sequences:
            for step in seq:
                tactics.append(step['tactic'])
                techniques.append(step['technique'])
                platforms.extend(step['platforms'])
                cmd_complexities.append(step['cmd_complexity'])
        
        # 训练编码器
        self.tactic_encoder.fit(list(set(tactics)))
        self.technique_encoder.fit(list(set(techniques)))
        self.platform_encoder.fit(list(set(platforms)))
        
        # 存储维度信息
        self.tactic_dim = len(self.tactic_encoder.classes_)
        self.technique_dim = len(self.technique_encoder.classes_)
        self.platform_dim = len(self.platform_encoder.classes_)
        self.feature_dim = self.tactic_dim + self.technique_dim + self.platform_dim + 1
        
    def transform_step(self, step):
        # 编码单个步骤
        tactic_idx = self.tactic_encoder.transform([step['tactic']])[0]
        technique_idx = self.technique_encoder.transform([step['technique']])[0]
        platform_idxs = self.platform_encoder.transform(step['platforms'])
        
        # 创建特征向量
        feature_vec = torch.zeros(self.feature_dim)
        feature_vec[tactic_idx] = 1  # tactic的one-hot
        feature_vec[self.tactic_dim + technique_idx] = 1  # technique的one-hot
        # 多个平台的multi-hot编码
        platform_offset = self.tactic_dim + self.technique_dim
        for idx in platform_idxs:
            feature_vec[platform_offset + idx] = 1
        # 添加命令复杂度
        feature_vec[-1] = step['cmd_complexity']
        
        return feature_vec

def pad_sequences_torch(sequences, max_length):
    # 获取特征维度
    feature_dim = sequences[0].size(1)
    
    # 创建填充后的张量
    padded = torch.zeros(len(sequences), max_length, feature_dim)
    
    # 填充序列
    for i, seq in enumerate(sequences):
        length = min(len(seq), max_length)
        padded[i, :length] = seq[:length]
    
    return padded


def build_features(sequences, max_length=50):
    # 创建和训练编码器
    encoder = FeatureEncoder()
    encoder.fit(sequences)
    
    # 编码序列
    encoded_seqs = []
    for seq in sequences:
        # 编码每个步骤
        encoded_steps = [encoder.transform_step(step) for step in seq]
        encoded_seq = torch.stack(encoded_steps)
        encoded_seqs.append(encoded_seq)
    
    # 填充序列
    padded_seqs = pad_sequences_torch(encoded_seqs, max_length)
    
    return padded_seqs, encoder



# 使用示例
def process_data(yaml_files):
    # 1. 解析YAML文件
    all_sequences = []
    for yaml_file in yaml_files:
        sequence = parse_attack_chain(yaml_file)
        all_sequences.append(sequence)
    
    # 2. 构建特征
    features, encoder = build_features(all_sequences)
    
    return features, encoder

In [ ]:
# 3.序列构建

def create_sequences(parsed_data):
    # 构建滑动窗口序列
    window_size = 5
    X, y = [], []
    
    for attack_chain in parsed_data:
        for i in range(len(attack_chain)-window_size):
            context = attack_chain[i:i+window_size]
            next_step = attack_chain[i+window_size]
            
            X.append([step['technique'] for step in context])  # T编号序列
            y.append(next_step['technique'])
    
    return np.array(X), np.array(y)

In [ ]:
import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size=128, num_layers=2):
        super(LSTMModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, 
                           hidden_size, 
                           num_layers=num_layers, 
                           batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        # 只取最后一个时间步的输出
        out = self.fc(lstm_out[:, -1, :])
        return torch.softmax(out, dim=1)

# 使用示例
def build_lstm_model(vocab_size, embedding_dim):
    return LSTMModel(vocab_size, embedding_dim)


In [ ]:
# 5.HMM适配

from hmmlearn import hmm

class AttackHMM:
    def __init__(self, n_states=10):
        self.model = hmm.GaussianHMM(n_components=n_states)
        
    def train(self, sequences):
        # 将攻击链转换为观测序列
        lengths = [len(seq) for seq in sequences]
        flattened = np.concatenate(sequences)
        self.model.fit(flattened.reshape(-1,1), lengths=lengths)
    
    def predict_next(self, current_seq):
        return self.model.predict(current_seq)[-1]

In [ ]:
# 6.LSTM训练
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.preprocessing import LabelEncoder

class AttackChainDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.LongTensor(X)
        self.y = torch.LongTensor(y)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def prepare_training_data(yaml_files):
    # 1. 解析所有YAML文件
    all_sequences = []
    for yaml_file in yaml_files:
        sequence = parse_attack_chain(yaml_file)
        all_sequences.append(sequence)
    
    # 2. 创建序列数据
    X, y = create_sequences(all_sequences)
    
    # 3. 对技术ID进行编码
    technique_encoder = LabelEncoder()
    # 将所有技术ID放在一起进行拟合
    all_techniques = np.concatenate([X.flatten(), y])
    technique_encoder.fit(all_techniques)
    
    # 转换数据
    X_encoded = np.array([technique_encoder.transform(seq) for seq in X])
    y_encoded = technique_encoder.transform(y)
    
    return X_encoded, y_encoded, technique_encoder

def train_model(model, train_loader, num_epochs, device='cuda'):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            # 计算准确率
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
            
        avg_loss = total_loss / len(train_loader)
        accuracy = 100 * correct / total
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%')

def main():
    # 配置参数
    yaml_files = ['path/to/your/yaml/files/*.yaml']  # 替换为实际的YAML文件路径
    window_size = 5  # 与create_sequences中的window_size保持一致
    batch_size = 32
    num_epochs = 10
    
    # 准备数据
    X_encoded, y_encoded, technique_encoder = prepare_training_data(yaml_files)
    vocab_size = len(technique_encoder.classes_)  # 词汇表大小为不同技术ID的数量
    embedding_dim = 100
    
    # 创建数据集和数据加载器
    dataset = AttackChainDataset(X_encoded, y_encoded)
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # 创建模型
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = LSTMModel(vocab_size=vocab_size, 
                     embedding_dim=embedding_dim, 
                     hidden_size=128, 
                     num_layers=2)
    
    # 训练模型
    train_model(model, train_loader, num_epochs, device=device)
    
    # 保存模型和编码器
    torch.save({
        'model_state_dict': model.state_dict(),
        'technique_encoder': technique_encoder
    }, 'attack_chain_model.pth')

if __name__ == "__main__":
    main()